In [1]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
test_file_names = ["full", "no_why", "no_question", "no_why_no_question"]

In [3]:
root_path = Path() / ".." / "data" / "experiments" / "bootstrap"

In [4]:
filename = test_file_names[0] + "_histogram_data.json"
file_path = root_path / filename

In [5]:
histo_data = pd.read_json(file_path, lines=False)

In [6]:
def confusion_matrix_np(y_true, y_pred):
    classes = np.unique(np.concatenate((y_true, y_pred)))
    conf_matrix = np.zeros((len(classes), len(classes)), dtype=int)
    class_to_index = {cls: idx for idx, cls in enumerate(classes)}
    for t, p in zip(y_true, y_pred):
        conf_matrix[class_to_index[t], class_to_index[p]] += 1
    return conf_matrix, classes

In [7]:
results_dir = Path() / ".." / "data" / "experiments" 
results_file = results_dir / "classification_results.csv"

df = pd.read_csv(results_file)

In [8]:
baseline_f1s = {model : {} for model in histo_data.keys()}
print(baseline_f1s)

baseline_f1s.pop('grok-4', None)
baseline_f1s

{'gpt-4o': {}, 'grok-3': {}, 'grok-4': {}, 'mistral-medium': {}}


{'gpt-4o': {}, 'grok-3': {}, 'mistral-medium': {}}

In [9]:
baseline_f1s = {model : {} for model in histo_data.keys()}
baseline_f1s.pop('grok-4', None)

for model_name, group in df.groupby("predictor"):
    if model_name == "grok-4":
        continue
    print(model_name)
    y_true = group["label"].to_numpy()
    y_pred = group["classification"].to_numpy()

    conf_matrix, classes = confusion_matrix_np(y_true, y_pred)

    TP = np.diag(conf_matrix)
    FP = np.sum(conf_matrix, axis=0) - TP
    FN = np.sum(conf_matrix, axis=1) - TP
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * precision * recall / (precision + recall)
    f1 = np.nan_to_num(f1, nan=0.0)

    support = np.sum(conf_matrix, axis=1)
    f1_weighted = np.sum(f1 * support) / np.sum(support)

    TP_micro = np.sum(TP)
    FP_micro = np.sum(FP)
    FN_micro = np.sum(FN)

    precision_micro = TP_micro / (TP_micro + FP_micro)
    recall_micro = TP_micro / (TP_micro + FN_micro)
    f1_micro = 2 * precision_micro * recall_micro / (precision_micro + recall_micro)


    baseline_f1s[model_name]["macro_f1"] = np.mean(f1).tolist()
    baseline_f1s[model_name]["micro_f1"] = f1_micro.tolist()
    # baseline_f1s[model_name]["weighted_f1"] = f1_weighted.tolist()


gpt-4o
grok-3
mistral-medium


C:\Users\Jordan\AppData\Local\Temp\ipykernel_16700\602851231.py:16: RuntimeWarning: invalid value encountered in divide
  precision = TP / (TP + FP)


In [10]:
baseline_f1s

{'gpt-4o': {'macro_f1': 0.6177963726745385, 'micro_f1': 0.6733333333333333},
 'grok-3': {'macro_f1': 0.5993924540408033, 'micro_f1': 0.6733333333333333},
 'mistral-medium': {'macro_f1': 0.6147430210890436, 'micro_f1': 0.67}}

In [11]:
histo_data

,gpt-4o,grok-3,grok-4,mistral-medium
macro_f1,"[0.594854202602734, 0.6277622836495851, 0.6244...","[0.592893244947795, 0.592277516108457, 0.59548...","[0.656453876077287, 0.6327897324575541, 0.6328...","[0.6256630475539601, 0.6132519864339351, 0.610..."
weighted_f1,"[0.592496112142575, 0.631344097301623, 0.63597...","[0.617245400599553, 0.6062587938321511, 0.5809...","[0.6531481726982971, 0.64601709156494, 0.65102...","[0.628374962312376, 0.6084282076704141, 0.6001..."
micro_f1,"[0.654889071487263, 0.686405337781484, 0.68861...","[0.6856649395509491, 0.6808688387635751, 0.658...","[0.7142857142857141, 0.702995008319467, 0.7071...","[0.680429397192402, 0.6666666666666661, 0.6589..."


In [12]:
model_rename = {
    "gpt-4o": "GPT 4o",
    "grok-3": "Grok 3",
    "mistral-medium": "Mistral Medium"
}
f1_rename = {
    "macro_f1": "Macro F1",
    "micro_f1": "Micro F1",
}

In [13]:
histo_data.drop(columns=["grok-4"], axis=1, inplace=True)

fig, axes = plt.subplots(len(histo_data.keys()), 2, figsize=(10,10), sharex=True, sharey=True, dpi=800)

f1_types = histo_data["gpt-4o"].keys().drop("weighted_f1")
alpha = 0.05
for i, (model, results) in enumerate(histo_data.items()):
    if model == "grok-4":
        continue

    count = 0
    for j, (f1_type, data) in enumerate(results.items()):
        if f1_type == "weighted_f1":
            continue
        lower = np.percentile(data, 100 * (alpha/2))
        upper = np.percentile(data, 100 * (1 - alpha/2))
        axes[i, count].axvspan(lower, upper, color='orange', alpha=0.3, label='95% CI (data range)')
        axes[i, count].hist(data, color='skyblue', edgecolor='black',bins=30, label="Counts of f1 score")
        #axes[i, j].set_title(f1_type)
        mean = np.mean(data)
        axes[i, count].axvline(mean, color='r', label="Mean f1 score", linewidth=4, alpha=0.6)
        axes[i, count].axvline(baseline_f1s[model][f1_type], color='g', label="Baseline f1 score", linewidth=1, alpha=0.6)
        # axes[i, j].text(mean, axes[i,j].get_ylim()[1]*0.9,
        #         f"μ={mean:.3f}\nσ={np.std(data):.3f}",
        #         color='red', ha='center left', fontsize=8)

        count += 1

for ax, col in zip(axes[0], f1_types):
    ax.set_title(f1_rename[col])

for ax, row in zip(axes[:,0], histo_data.keys()):
    ax.set_ylabel(model_rename[row], size='large')

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(),loc='center left', bbox_to_anchor=(1, 0.5))
fig.tight_layout(rect=[0, 0, 1, 1])  # Make room for the legend

plt.show()

In [15]:
ablation_rename = {
    "no_why" : "No Why",
    "no_question" : "No Question",
    "no_why_no_question" : "No Why & No Question"
}


fig, axes = plt.subplots(len(test_file_names)-1, 2, figsize=(10,10), sharex=True, sharey=True, dpi=800)

for i in range(1, len(test_file_names)):
    print(test_file_names[i])
    filename = test_file_names[i] + "_histogram_data.json"
    file_path = root_path / filename

    histo_data = pd.read_json(file_path, lines=False)

    results_dir = Path() / ".." / "data" / "experiments" 
    results_file = results_dir / "classification_results.csv"

    df = pd.read_csv(results_file)

    baseline_f1s = {model : {} for model in histo_data.keys()}
    baseline_f1s.pop('grok-4', None)

    for model_name, group in df.groupby("predictor"):
        if model_name == "grok-4":
            continue
        print(model_name)
        y_true = group["label"].to_numpy()
        y_pred = group["classification"].to_numpy()

        conf_matrix, classes = confusion_matrix_np(y_true, y_pred)

        TP = np.diag(conf_matrix)
        FP = np.sum(conf_matrix, axis=0) - TP
        FN = np.sum(conf_matrix, axis=1) - TP
        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        f1 = 2 * precision * recall / (precision + recall)
        f1 = np.nan_to_num(f1, nan=0.0)

        support = np.sum(conf_matrix, axis=1)
        f1_weighted = np.sum(f1 * support) / np.sum(support)

        TP_micro = np.sum(TP)
        FP_micro = np.sum(FP)
        FN_micro = np.sum(FN)

        precision_micro = TP_micro / (TP_micro + FP_micro)
        recall_micro = TP_micro / (TP_micro + FN_micro)
        f1_micro = 2 * precision_micro * recall_micro / (precision_micro + recall_micro)


        baseline_f1s[model_name]["macro_f1"] = np.mean(f1).tolist()
        baseline_f1s[model_name]["micro_f1"] = f1_micro.tolist()
        baseline_f1s[model_name]["weighted_f1"] = f1_weighted.tolist()


    histo_data.drop(columns=["grok-4"], axis=1, inplace=True)

    

    f1_types = histo_data["gpt-4o"].keys().drop("weighted_f1")
    alpha = 0.05

    for __annotations__, (model, results) in enumerate(histo_data.items()):
        if model == "gpt-4o":
            count = 0
            for j, (f1_type, data) in enumerate(iterable=results.items()):
                if f1_type == "weighted_f1":
                    continue
                lower = np.percentile(data, 100 * (alpha/2))
                upper = np.percentile(data, 100 * (1 - alpha/2))
                axes[i-1, count].axvspan(lower, upper, color='orange', alpha=0.3, label='95% CI (data range)')
                axes[i-1, count].hist(data, color='skyblue', edgecolor='black',bins=30, label="Counts of f1 score")
                # axes[i-1, j].set_title(test_file_names[i])
                mean = np.mean(data)
                axes[i-1, count].axvline(mean, color='r', label="Mean f1 score", linewidth=4, alpha=0.6)
                axes[i-1, count].axvline(baseline_f1s[model][f1_type], color='g', label="Baseline f1 score", linewidth=1, alpha=0.6)
                # axes[i, j].text(mean, axes[i,j].get_ylim()[1]*0.9,
                #         f"μ={mean:.3f}\nσ={np.std(data):.3f}",
                #         color='red', ha='center left', fontsize=8)
                count += 1

for ax, col in zip(axes[0], f1_types):
    ax.set_title(f1_rename[col])

for ax, row in zip(axes[:,0], np.arange(1,len(test_file_names))):
    print(row)
    ax.set_ylabel(ablation_rename[test_file_names[row]], size='large')

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(),loc='center left', bbox_to_anchor=(1, 0.5))
fig.tight_layout(rect=[0, 0, 1, 1])  # Make room for the legend

plt.show()



no_why
gpt-4o
grok-3
mistral-medium
no_question
gpt-4o
grok-3
mistral-medium


C:\Users\Jordan\AppData\Local\Temp\ipykernel_16700\1604238129.py:37: RuntimeWarning: invalid value encountered in divide
  precision = TP / (TP + FP)
C:\Users\Jordan\AppData\Local\Temp\ipykernel_16700\1604238129.py:37: RuntimeWarning: invalid value encountered in divide
  precision = TP / (TP + FP)


no_why_no_question
gpt-4o
grok-3
mistral-medium
1
2
3


C:\Users\Jordan\AppData\Local\Temp\ipykernel_16700\1604238129.py:37: RuntimeWarning: invalid value encountered in divide
  precision = TP / (TP + FP)
